# Pretrain Variant 3 — From Scratch (gate init = -1.0)

Checkpoint cũ có gate values đã converge về vùng saturated (sigmoid(-5) ≈ 0),
load lại sẽ không có tác dụng. Cần retrain from scratch với init mới (-1.0).

Model: `FusionEncoderNoFilterVideoRoPELG`
- NoFilter (uniform pooling)
- VideoRoPE 2D (temporal LTA + spatial axial 4×4)
- Local-Global Cross-Attention
- L = 2 fusion blocks

In [1]:
import gc
import json
import math
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

cwd = Path.cwd().resolve()
if (cwd / 'source').exists():
    BASE_DIR = cwd
elif (cwd.parent / 'source').exists():
    BASE_DIR = cwd.parent
else:
    BASE_DIR = Path('/media/urlab/KINGSTON/aic')

SOURCE_DIR = BASE_DIR / 'source'
sys.path.insert(0, str(SOURCE_DIR / 'train'))
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('BASE_DIR:', BASE_DIR)
print('DEVICE  :', DEVICE)
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

/home/urlab/miniconda3/envs/uav_ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE_DIR: /media/urlab/KINGSTON/aic
DEVICE  : cuda
GPU     : NVIDIA GeForce RTX 5060 Ti


In [2]:
from dataset import get_dataloader
from model import (
    FusionEncoderNoFilterVideoRoPELG,
    build_fusion_encoder,
    FusionConfig,
)

OUTPUT_DIR = SOURCE_DIR / 'pretrain_v3_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BGEM3_PATH = BASE_DIR / 'data' / 'features' / 'weights' / 'bgem3'

CFG = {
    'base_dir': BASE_DIR,
    'max_frames': 25,
    'max_segments': 15,
    'max_seg_tokens': 128,
    'dim': 1024,
    'vis_dim': 1152,
    'n_heads': 16,
    'n_kv_heads': 4,
    'epochs': 75,
    'batch_size': 2,
    'grad_accum_steps': 8,
    'num_workers': os.cpu_count() or 4,
    'lr': 1e-4,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'grad_clip': 1.0,
    'train_query_max_length': 512,
    'dual_softmax_tau': 0.01,
    'eval_query_max_length': 256,
    'eval_query_batch_size': 16,
    'visual_feature_subdir': 'siglip2',
    # VideoRoPE config
    'rope_base_temporal': 500_000.0,
    'rope_base_spatial': 10_000.0,
    'delta': 100.0,
    # Local-Global CA config
    'lg_window_seconds': 8.0,
    'lg_n_global_tokens': 16,
}
print('Output dir:', OUTPUT_DIR)

Output dir: /media/urlab/KINGSTON/aic/source/pretrain_v3_output


In [3]:
train_loader = get_dataloader(
    split='train', base_dir=CFG['base_dir'],
    batch_size=CFG['batch_size'], num_workers=CFG['num_workers'],
    max_frames=CFG['max_frames'], max_segments=CFG['max_segments'],
    max_seg_tokens=CFG['max_seg_tokens'],
    visual_feature_subdir=CFG['visual_feature_subdir'],
)
val_loader = get_dataloader(
    split='val', base_dir=CFG['base_dir'],
    batch_size=CFG['batch_size'], num_workers=CFG['num_workers'],
    max_frames=CFG['max_frames'], max_segments=CFG['max_segments'],
    max_seg_tokens=CFG['max_seg_tokens'],
    visual_feature_subdir=CFG['visual_feature_subdir'],
)

bgem3_tokenizer = AutoTokenizer.from_pretrained(str(BGEM3_PATH), local_files_only=True)
bgem3_model = AutoModel.from_pretrained(str(BGEM3_PATH), local_files_only=True)
bgem3_model.eval().to(DEVICE)
for p in bgem3_model.parameters():
    p.requires_grad = False

print('Train samples:', len(train_loader.dataset))
print('Val   samples:', len(val_loader.dataset))

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 23657.66it/s]


Train samples: 11328
Val   samples: 11334


In [4]:
def encode_queries(texts, tokenizer, encoder_model, device, max_length=512, batch_size=None):
    if not texts:
        return torch.empty((0, encoder_model.config.hidden_size), device=device)
    if batch_size is None:
        batch_size = len(texts)
    all_embs = []
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            chunk = texts[start:start + batch_size]
            enc = tokenizer(chunk, padding=True, truncation=True,
                            max_length=max_length, return_tensors='pt').to(device)
            if device.type == 'cuda':
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    h = encoder_model(**enc, return_dict=True).last_hidden_state
            else:
                h = encoder_model(**enc, return_dict=True).last_hidden_state
            m = enc['attention_mask'].unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(1) / m.sum(1).clamp_min(1e-6)
            all_embs.append(F.normalize(pooled.float(), p=2, dim=-1))
    return torch.cat(all_embs, dim=0)


def symmetric_infonce(q, d, temperature):
    tau = torch.exp(temperature).clamp_min(1e-6)
    sim = torch.matmul(q, d.T) / tau
    labels = torch.arange(sim.size(0), device=sim.device)
    return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels))


def retrieval_loss(q_hat, e_plus, e_narr, temperature, hard_neg_embs=None,
                   lambda_narr=0.5, beta_hard=0.3, gamma_mrl=0.1):
    l_base = symmetric_infonce(q_hat, e_plus, temperature)
    l_narr = torch.tensor(0.0, device=q_hat.device)
    if lambda_narr > 0:
        l_narr = symmetric_infonce(q_hat, e_narr, temperature)
    l_hard = torch.tensor(0.0, device=q_hat.device)
    if hard_neg_embs is not None and beta_hard > 0:
        q = q_hat.unsqueeze(1)
        neg_scores = torch.sum(q * hard_neg_embs, dim=-1)
        pos_scores = torch.sum(q_hat * e_plus, dim=-1, keepdim=True)
        l_hard = torch.relu(neg_scores - pos_scores + 0.1).mean()
    l_mrl = torch.tensor(0.0, device=q_hat.device)
    if gamma_mrl > 0:
        for d_dim in [64, 128, 256, 512, 1024]:
            l_mrl = l_mrl + symmetric_infonce(q_hat[:, :d_dim], e_plus[:, :d_dim], temperature)
        l_mrl = l_mrl / 5.0
    return l_base + lambda_narr * l_narr + beta_hard * l_hard + gamma_mrl * l_mrl


def _encode_hard_negs(hard_neg_texts_batch, max_length):
    b = len(hard_neg_texts_batch)
    k = max((len(x) for x in hard_neg_texts_batch), default=0)
    if k == 0:
        return None
    flat = []
    for negs in hard_neg_texts_batch:
        flat.extend(negs + [''] * (k - len(negs)))
    emb = encode_queries(flat, bgem3_tokenizer, bgem3_model, DEVICE, max_length=max_length)
    return emb.view(b, k, -1)


def precompute_val_docs(model, loader, device):
    model.eval()
    doc_embs, shot_ids = [], []
    with torch.inference_mode():
        for batch in tqdm(loader, desc='Val docs', leave=False, dynamic_ncols=True):
            with torch.amp.autocast(device_type='cuda' if device.type == 'cuda' else 'cpu',
                                   enabled=(device.type == 'cuda'), dtype=torch.bfloat16):
                e_plus, _, _ = model(
                    seg_tokens=batch['token_reprs'].to(device),
                    seg_pooled=batch['segment_pooled'].to(device),
                    visual_features=batch['visual_features'].to(device),
                    seg_timestamps=batch['seg_timestamps'].to(device),
                    frame_timestamps=batch['frame_timestamps'].to(device),
                    seg_mask=batch['segment_mask'].to(device),
                    visual_mask=batch['visual_mask'].to(device),
                    query_emb=None,
                    token_mask=batch['token_mask'].to(device),
                )
            doc_embs.append(e_plus.detach().cpu())
            shot_ids.extend(batch['shot_id'])
    return F.normalize(torch.cat(doc_embs, dim=0), p=2, dim=-1), shot_ids


def load_val_queries():
    q_items = []
    for jf in sorted((BASE_DIR / 'data' / 'val').glob('*.json')):
        video_id = jf.stem
        for row in json.loads(jf.read_text('utf-8')):
            sid = str(row.get('id', '')).zfill(3)
            q = str(row.get('positive', '')).strip()
            if sid and q:
                q_items.append({'shot_id': f'{video_id}_{sid}', 'query': q})
    seen, deduped = set(), []
    for x in q_items:
        if x['shot_id'] not in seen:
            seen.add(x['shot_id'])
            deduped.append(x)
    return deduped


def evaluate_on_val(model, device):
    doc_embs, shot_ids = precompute_val_docs(model, val_loader, device)
    q_items = load_val_queries()
    q_embs_all = encode_queries(
        [x['query'] for x in q_items],
        bgem3_tokenizer, bgem3_model, device,
        max_length=CFG['eval_query_max_length'],
        batch_size=CFG['eval_query_batch_size'],
    )
    shot_to_idx = {s: i for i, s in enumerate(shot_ids)}
    filtered = [x for x in q_items if x['shot_id'] in shot_to_idx]
    q_idx = [q_items.index(x) for x in filtered]
    q_embs_f = q_embs_all[q_idx]
    gt_indices = [shot_to_idx[x['shot_id']] for x in filtered]
    tau = CFG['dual_softmax_tau']
    sim_raw = torch.matmul(q_embs_f, doc_embs.to(device).T) / tau
    sim_dsl = torch.softmax(sim_raw, dim=1) * torch.softmax(sim_raw, dim=0)
    sim_np = sim_dsl.detach().cpu().numpy()
    ranks = [int(np.where(np.argsort(-sim_np[i]) == gt)[0][0]) + 1
             for i, gt in enumerate(gt_indices)]
    ranks = np.array(ranks)
    return {
        'R1':  float((ranks <= 1).mean() * 100),
        'R5':  float((ranks <= 5).mean() * 100),
        'R10': float((ranks <= 10).mean() * 100),
        'SumR': float(((ranks <= 1).mean() + (ranks <= 5).mean() + (ranks <= 10).mean()) * 100),
        'MedR': float(np.median(ranks)),
        'MeanR': float(ranks.mean()),
        'n_queries': len(ranks),
    }

## Build & Train Variant 3 from scratch

Gate init = -1.0 → sigmoid(-1) ≈ 0.27 (unsaturated, gradients flow)
Old init = -5.0 → sigmoid(-5) ≈ 0.007 (saturated, near-zero output)

In [5]:
loss_cfg = dict(lambda_narr=0.5, beta_hard=0.3, gamma_mrl=0.1)

model = FusionEncoderNoFilterVideoRoPELG(
    dim=CFG['dim'], vis_dim=CFG['vis_dim'],
    n_layers=2, n_heads=CFG['n_heads'], n_kv_heads=CFG['n_kv_heads'],
    delta=CFG['delta'],
    base_temporal=CFG['rope_base_temporal'],
    base_spatial=CFG['rope_base_spatial'],
    window_seconds=CFG['lg_window_seconds'],
    n_global_tokens=CFG['lg_n_global_tokens'],
).to(DEVICE)

# Verify gate init is -1.0 (not old -5.0)
for name, param in model.named_parameters():
    if 'gate' in name:
        print(f'{name}: mean={param.data.mean().item():.2f}, '
              f'sigmoid_mean={torch.sigmoid(param.data).mean().item():.4f}')

print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.2f} M')

blocks.0.self_attn.gate: mean=-1.00, sigmoid_mean=0.2689
blocks.0.ffn1.w_gate.weight: mean=-0.00, sigmoid_mean=0.5000
blocks.0.local_ca.gate: mean=-1.00, sigmoid_mean=0.2689
blocks.0.global_ca.gate: mean=-1.00, sigmoid_mean=0.2689
blocks.0.ffn2.w_gate.weight: mean=-0.00, sigmoid_mean=0.5000
blocks.1.self_attn.gate: mean=-1.00, sigmoid_mean=0.2689
blocks.1.ffn1.w_gate.weight: mean=-0.00, sigmoid_mean=0.5000
blocks.1.local_ca.gate: mean=-1.00, sigmoid_mean=0.2689
blocks.1.global_ca.gate: mean=-1.00, sigmoid_mean=0.2689
blocks.1.ffn2.w_gate.weight: mean=-0.00, sigmoid_mean=0.5000
Params: 50.47 M


In [6]:
accum_steps = max(1, int(CFG.get('grad_accum_steps', 1)))
steps_per_epoch = math.ceil(len(train_loader) / accum_steps)
total_steps = max(1, steps_per_epoch * CFG['epochs'])
warmup_steps = int(total_steps * CFG['warmup_ratio'])

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler = torch.amp.GradScaler('cuda' if DEVICE.type == 'cuda' else 'cpu')

ckpt_name = 'pretrain_v3_gate_neg1.pth'
ckpt_path = OUTPUT_DIR / ckpt_name
last_path = OUTPUT_DIR / 'last.pt'
log_path  = OUTPUT_DIR / ckpt_name.replace('.pth', '_log.json')

print(f'Total steps: {total_steps}, warmup: {warmup_steps}')
print(f'Checkpoint:  {ckpt_path}')

Total steps: 53100, warmup: 5310
Checkpoint:  /media/urlab/KINGSTON/aic/source/pretrain_v3_output/pretrain_v3_gate_neg1.pth


In [7]:
history = []
best_r1 = -1.0
DESC = 'V3-gate-1.0'

for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    running = 0.0
    pbar = tqdm(train_loader, desc=f'[{DESC}] ep{epoch:02d}', leave=False, dynamic_ncols=True)
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(pbar, start=1):
        q_hat = encode_queries(batch['query_text'], bgem3_tokenizer, bgem3_model, DEVICE,
                                max_length=CFG['train_query_max_length'])
        hard_neg = _encode_hard_negs(batch['hard_neg_texts'], CFG['train_query_max_length'])

        with torch.amp.autocast(device_type='cuda' if DEVICE.type == 'cuda' else 'cpu',
                               enabled=(DEVICE.type == 'cuda'), dtype=torch.bfloat16):
            e_plus, e_narr, temperature = model(
                seg_tokens=batch['token_reprs'].to(DEVICE),
                seg_pooled=batch['segment_pooled'].to(DEVICE),
                visual_features=batch['visual_features'].to(DEVICE),
                seg_timestamps=batch['seg_timestamps'].to(DEVICE),
                frame_timestamps=batch['frame_timestamps'].to(DEVICE),
                seg_mask=batch['segment_mask'].to(DEVICE),
                visual_mask=batch['visual_mask'].to(DEVICE),
                query_emb=q_hat,
                token_mask=batch['token_mask'].to(DEVICE),
            )
            loss = retrieval_loss(q_hat, e_plus, e_narr, temperature,
                                  hard_neg_embs=hard_neg, **loss_cfg)

        loss_raw = loss.detach()
        loss = loss / accum_steps
        scaler.scale(loss).backward()

        if step % accum_steps == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(optimizer)
            with torch.no_grad():
                model.temperature.clamp_(min=math.log(0.01), max=math.log(1.0))
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        running += loss_raw.item()
        pbar.set_postfix(loss=f'{running / (pbar.n + 1):.4f}')

    train_loss = running / max(1, len(train_loader))

    if DEVICE.type == 'cuda':
        gc.collect(); torch.cuda.empty_cache()

    val_metrics = evaluate_on_val(model, DEVICE)
    history.append({'epoch': epoch, 'train_loss': train_loss, **val_metrics})

    if val_metrics['R1'] > best_r1:
        best_r1 = val_metrics['R1']
        torch.save(model.state_dict(), ckpt_path)
    torch.save(model.state_dict(), last_path)
    log_path.write_text(json.dumps(history, indent=2), encoding='utf-8')
    print(f"[{DESC}] ep{epoch:02d} loss={train_loss:.4f} R@1={val_metrics['R1']:.2f} "
          f"R@5={val_metrics['R5']:.2f} R@10={val_metrics['R10']:.2f}")

print(f'[{DESC}] DONE | best R@1={best_r1:.2f} | ckpt={ckpt_path}')

[V3-gate-1.0] ep01 loss=1.0681 R@1=22.46 R@5=40.78 R@10=49.72


[V3-gate-1.0] ep02 loss=1.0158 R@1=27.57 R@5=48.84 R@10=58.24


[V3-gate-1.0] ep03 loss=0.9940 R@1=34.03 R@5=56.35 R@10=65.40


[V3-gate-1.0] ep04 loss=0.9796 R@1=40.05 R@5=62.91 R@10=71.05


[V3-gate-1.0] ep05 loss=0.9649 R@1=45.63 R@5=68.55 R@10=75.90


...

[V3-gate-1.0] ep71 loss=0.1417 R@1=73.84 R@5=89.69 R@10=93.36


[V3-gate-1.0] ep72 loss=0.1416 R@1=73.80 R@5=89.64 R@10=93.32


[V3-gate-1.0] ep73 loss=0.1412 R@1=73.82 R@5=89.65 R@10=93.33


[V3-gate-1.0] ep74 loss=0.1392 R@1=73.81 R@5=89.66 R@10=93.32


[V3-gate-1.0] ep75 loss=0.1415 R@1=73.80 R@5=89.65 R@10=93.33
[V3-gate-1.0] DONE | best R@1=73.84 | ckpt=/media/urlab/KINGSTON/aic/source/pretrain_v3_output/pretrain_v3_gate_neg1.pth
